# Optimize Weld's Experiment with Ax


In [15]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

In [16]:
NUM_NEW_EXP = 3

In [17]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [18]:
folder_path = Path(os.getcwd()) / "data"

In [19]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,1,205.512265,204.738760,207.056342,183.713911,181.792203,184.483996
1,1,1462,1811,1851,93,63,41,17,1,199.002716,200.430817,198.957274,170.864008,177.489548,176.112604
2,2,1821,1598,1421,62,74,99,20,1,210.138445,208.003040,208.332095,180.967275,193.690799,191.724174
3,3,1894,1873,1756,73,98,82,14,1,218.672547,204.882794,213.804086,189.691174,190.036535,190.747691
4,4,1691,1519,1622,51,41,57,29,1,202.873540,204.304761,209.820085,185.025997,178.124362,183.032944
5,5,1877,1560,1515,45,73,100,24,1,209.429590,207.862307,211.090300,191.923736,194.070540,190.991520


In [20]:
print(exp.columns)
cols_advancing = [
    "sigma_advancing_1",
    "sigma_advancing_2",
    "sigma_advancing_3",
]
cols_retreating = [
    "sigma_retreating_1",
    "sigma_retreating_2",
    "sigma_retreating_3",
]

Index(['trial_index', 'R1', 'R2', 'R3', 'W1', 'W2', 'W3', 'D',
       'visual_quality', 'sigma_advancing_1', 'sigma_advancing_2',
       'sigma_advancing_3', 'sigma_retreating_1', 'sigma_retreating_2',
       'sigma_retreating_3'],
      dtype='str')


In [21]:
filt_advancing = exp[cols_advancing].isna().all(axis=1)
filt_retreating = exp[cols_retreating].isna().all(axis=1)
filt_removed = filt_advancing | filt_retreating
filt_removed

0    False
1    False
2    False
3    False
4    False
5    False
dtype: bool

In [22]:
exp_todo = exp[filt_removed].copy()
exp = exp[~filt_removed].copy()
display(exp)
display(exp_todo)

,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,1,205.512265,204.738760,207.056342,183.713911,181.792203,184.483996
1,1,1462,1811,1851,93,63,41,17,1,199.002716,200.430817,198.957274,170.864008,177.489548,176.112604
2,2,1821,1598,1421,62,74,99,20,1,210.138445,208.003040,208.332095,180.967275,193.690799,191.724174
3,3,1894,1873,1756,73,98,82,14,1,218.672547,204.882794,213.804086,189.691174,190.036535,190.747691
4,4,1691,1519,1622,51,41,57,29,1,202.873540,204.304761,209.820085,185.025997,178.124362,183.032944
5,5,1877,1560,1515,45,73,100,24,1,209.429590,207.862307,211.090300,191.923736,194.070540,190.991520


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3


In [23]:
# 2. Configure where Ax will search.

bound_R = [1400, 2000]  # Rotatinal speed
bound_W = [40, 100]  # Welding speed
bound_D = [10, 30]  # Dwell time

parameters = [
    RangeParameterConfig(
        name="R1",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R2",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R3",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W1",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W2",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W3",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="D",
        bounds=bound_D,
        parameter_type="int",
    ),
]

client.configure_experiment(
    name="parn_exp",
    parameters=parameters,
)

In [24]:
client.configure_optimization(
    objective="sigma_advancing, sigma_retreating",
    outcome_constraints=["visual_quality >= 0.5"],
)

In [25]:
def cal_mean_sem(row, cols):
    filt = row[cols].notna()
    arr = row[cols][filt].astype(float).values
    mean = arr.mean()
    sem = arr.std() / np.sqrt(len(arr))
    return mean, sem

In [26]:
for idx, row in exp.iterrows():
    trial_index = client.attach_trial(
        parameters={
            "R1": int(row["R1"]),
            "R2": int(row["R2"]),
            "R3": int(row["R3"]),
            "W1": int(row["W1"]),
            "W2": int(row["W2"]),
            "W3": int(row["W3"]),
            "D": int(row["D"]),
        }
    )
    advancing_mean, advancing_sem = cal_mean_sem(row, cols_advancing)
    retreating_mean, retreating_sem = cal_mean_sem(row, cols_retreating)
    visual_quality = float(row["visual_quality"])
    print(f"Trial {trial_index}: mean={advancing_mean}, sem={advancing_sem}")
    print(f"Trial {trial_index}: mean={retreating_mean}, sem={retreating_sem}")
    print(f"Trial {trial_index}: visual_quality={visual_quality}")
    client.complete_trial(
        trial_index=trial_index,
        raw_data={
            "sigma_advancing": (advancing_mean, advancing_sem),
            "sigma_retreating": (retreating_mean, retreating_sem),
            "visual_quality": visual_quality,
        },
    )

[INFO 07-22 09:49:37] ax.api.client: Trial 0 marked COMPLETED.
[INFO 07-22 09:49:37] ax.api.client: Trial 1 marked COMPLETED.
[INFO 07-22 09:49:37] ax.api.client: Trial 2 marked COMPLETED.
[INFO 07-22 09:49:37] ax.api.client: Trial 3 marked COMPLETED.
[INFO 07-22 09:49:37] ax.api.client: Trial 4 marked COMPLETED.
[INFO 07-22 09:49:37] ax.api.client: Trial 5 marked COMPLETED.


Trial 0: mean=205.76912250064527, sem=0.5562331606579651
Trial 0: mean=183.33003664342468, sem=0.6535301617298568
Trial 0: visual_quality=1.0
Trial 1: mean=199.46360236087176, sem=0.39500911336789224
Trial 1: mean=174.82205335559163, sem=1.6481358628144709
Trial 1: visual_quality=1.0
Trial 2: mean=208.82452664375742, sem=0.5419832113267186
Trial 2: mean=188.794082691231, sem=3.2287285371390504
Trial 2: visual_quality=1.0
Trial 3: mean=212.4531422642889, sem=3.2967359169855652
Trial 3: mean=190.1584666502715, sem=0.2539500135907286
Trial 3: visual_quality=1.0
Trial 4: mean=205.6661286329495, sem=1.7290726034447788
Trial 4: mean=182.0611007893424, sem=1.6744153881636594
Trial 4: visual_quality=1.0
Trial 5: mean=209.4607324117773, sem=0.7609512557989363
Trial 5: mean=192.3285988477062, sem=0.7443157724685157
Trial 5: visual_quality=1.0


In [27]:
if exp_todo.shape[0] == 0:
    suggest_params = []
    for i in range(NUM_NEW_EXP):
        data = client.get_next_trials(max_trials=1)
        for trial_index, parameters in data.items():
            print(f"Trial {trial_index} with parameters {parameters}")
            suggest_params.append({"trial_index": trial_index, **parameters})
    _exp = pd.DataFrame(suggest_params)
    _exp[cols_advancing + cols_retreating] = pd.NA
    exp = pd.concat([exp, _exp], ignore_index=True).reset_index(drop=True)
    display(exp)
    SAVE_EXCEL = True
else:
    print("There are still experiment to be conducted. Skip generating new sets")
    SAVE_EXCEL = False

[INFO 07-22 09:49:37] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol', use_existing_trials_for_initialization=True), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 07-22 09:49:37] ax.api.client: Generated new trial 6 with parameters {'R1': 1561, 'R2': 1943, 'R3': 1492, 'W1': 42, 'W2': 48, 'W3': 47, 'D': 12} using GenerationNode Sobol.
c:\Users\admin\Coding\research\weld-ml-v2\.venv\

Trial 6 with parameters {'R1': 1561, 'R2': 1943, 'R3': 1492, 'W1': 42, 'W2': 48, 'W3': 47, 'D': 12}


c:\Users\admin\Coding\research\weld-ml-v2\.venv\Lib\site-packages\botorch\models\utils\assorted.py:279: InputDataWarning: Data (outcome observations) is not standardized (std = tensor([0.], dtype=torch.float64), mean = tensor([-1.1102e-16], dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
  check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)
c:\Users\admin\Coding\research\weld-ml-v2\.venv\Lib\site-packages\ax\generators\torch\botorch_moo_utils.py:237: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  obj_mask = torch.tensor(obj_indices, device=objective_weights.device)
W0722 09:49:37.921000 24644 Lib\site-packages\torch\utils\cpp_extension.py:606] Error checking compiler version for cl
W0722 09:49:37.921000 24644 Lib\site-packages\torch\utils\cpp_extension.py:606] Traceback (most recent call 

Trial 7 with parameters {'R1': 2000, 'R2': 1791, 'R3': 1633, 'W1': 45, 'W2': 99, 'W3': 100, 'D': 18}


[INFO 07-22 09:49:56] ax.api.client: Generated new trial 8 with parameters {'R1': 2000, 'R2': 1412, 'R3': 1804, 'W1': 61, 'W2': 66, 'W3': 100, 'D': 16} using GenerationNode MBM.


Trial 8 with parameters {'R1': 2000, 'R2': 1412, 'R3': 1804, 'W1': 61, 'W2': 66, 'W3': 100, 'D': 16}


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,1.0,205.512265,204.73876,207.056342,183.713911,181.792203,184.483996
1,1,1462,1811,1851,93,63,41,17,1.0,199.002716,200.430817,198.957274,170.864008,177.489548,176.112604
2,2,1821,1598,1421,62,74,99,20,1.0,210.138445,208.00304,208.332095,180.967275,193.690799,191.724174
3,3,1894,1873,1756,73,98,82,14,1.0,218.672547,204.882794,213.804086,189.691174,190.036535,190.747691
4,4,1691,1519,1622,51,41,57,29,1.0,202.87354,204.304761,209.820085,185.025997,178.124362,183.032944
5,5,1877,1560,1515,45,73,100,24,1.0,209.42959,207.862307,211.0903,191.923736,194.07054,190.99152
6,6,1561,1943,1492,42,48,47,12,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,7,2000,1791,1633,45,99,100,18,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,8,2000,1412,1804,61,66,100,16,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [28]:
if SAVE_EXCEL:
    exp.to_excel(folder_path / "experiments.xlsx", index=False)